# GraphBench: Graph-Based RAG Benchmark

End-to-end notebook for all phases: data ingestion, GNN training, benchmark, and results.

**Runtime**: Colab Pro (GPU, High-RAM)  
**Checkpoints**: Google Drive at `/content/drive/MyDrive/graphbench/`

| Phase | Description |
|-------|-------------|
| 1 | Setup & Dependencies |
| 2 | Data Pipeline: REBEL Triples → FAISS + Neo4j |
| 3 | GNN Training: 3-Layer GAT |
| 4 | Pipelines: GraphRAG & GNN-RAG |
| 5 | Benchmark: 500 HotpotQA Questions |
| 6 | Results & Analysis |
| 7 | Export for Demo |

## Phase 1 — Setup & Dependencies

In [46]:
!pip install git+https://github.com/Rohanjain2312/graphbench.git -q --force-reinstall

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 16.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 8.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 11.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 8.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 10.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

In [1]:
# Install graphbench-kg from PyPI (or from source for development)
#!pip install graphbench-kg -q
# Alternatively, install from GitHub main:
!pip install git+https://github.com/Rohanjain2312/graphbench.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [47]:
# Mount Google Drive for persistent storage
from google.colab import drive

drive.mount("/content/drive")

import os
from pathlib import Path

DRIVE_BASE = Path("/content/drive/MyDrive/graphbench")
DRIVE_BASE.mkdir(parents=True, exist_ok=True)

# Sub-directories
CHECKPOINT_DIR = DRIVE_BASE / "checkpoints"
RESULTS_DIR = DRIVE_BASE / "results"
FAISS_PATH = DRIVE_BASE / "faiss_index"
for d in [CHECKPOINT_DIR, RESULTS_DIR]:
    d.mkdir(exist_ok=True)

print("Drive mounted. Base dir:", DRIVE_BASE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. Base dir: /content/drive/MyDrive/graphbench


In [48]:
# Load environment variables from Drive .env file
# Copy your .env to /content/drive/MyDrive/graphbench/.env before running
from dotenv import load_dotenv

env_path = DRIVE_BASE / ".env"
if env_path.exists():
    load_dotenv(env_path)
    print(f".env loaded from {env_path}")
else:
    print(f"WARNING: No .env found at {env_path}")
    print(
        "Set NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD, HF_TOKEN, WANDB_API_KEY manually:"
    )
    # Uncomment and fill in if no .env file:
    # import os
    # os.environ['NEO4J_URI']      = 'neo4j+s://YOUR_AURADB_URI'
    # os.environ['NEO4J_USERNAME'] = 'neo4j'
    # os.environ['NEO4J_PASSWORD'] = 'YOUR_PASSWORD'
    # os.environ['HF_TOKEN']       = 'hf_YOUR_TOKEN'
    # os.environ['WANDB_API_KEY']  = 'YOUR_WANDB_KEY'

.env loaded from /content/drive/MyDrive/graphbench/.env


In [49]:
# Verify installation and GPU
from graphbench import __version__
import torch

print(f"graphbench-kg version : {__version__}")
print(f"PyTorch version       : {torch.__version__}")
print(f"CUDA available        : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU                   : {torch.cuda.get_device_name(0)}")
    print(
        f"VRAM                  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB"
    )

graphbench-kg version : 0.1.0
PyTorch version       : 2.10.0+cu128
CUDA available        : True
GPU                   : NVIDIA H100 80GB HBM3
VRAM                  : 85.0 GB


## Phase 2 — Data Pipeline: REBEL Triples → FAISS + Neo4j

In [50]:
# Initialise Neo4j client and verify connectivity
from graphbench.utils.neo4j_client import Neo4jClient

neo4j = Neo4jClient()
neo4j.ensure_schema()
print(f"Neo4j connected. Existing triples: {neo4j.count_triples():,}")

Neo4j connected. Existing triples: 11,493


In [51]:
from graphbench.ingestion.rebel_loader import load_hotpotqa_passages
from itertools import islice

MAX_PASSAGES = 40_000  # ~1.5–2 hrs on GPU, yields ~40–80k triples

print("Streaming HotpotQA passages (capped)...")

passages = list(
    islice(
        load_hotpotqa_passages(split="train"),
        MAX_PASSAGES
    )
)

texts = [p["text"] for p in passages]

print(f"Collected {len(texts):,} passages")

Streaming HotpotQA passages (capped)...
Collected 40,000 passages


In [18]:
from graphbench.ingestion.triple_extractor import extract_from_passages
from tqdm import tqdm

all_triples = []
BATCH_SIZE = 8

passage_batches = [
    texts[i:i + BATCH_SIZE]
    for i in range(0, len(texts), BATCH_SIZE)
]

print(
    f"Running REBEL on {len(passage_batches):,} batches "
    f"(~{len(passage_batches) * 1.4 / 3600:.1f} hrs)..."
)

for batch in tqdm(passage_batches, desc="REBEL batches"):
    for triple in extract_from_passages(
        batch,
        batch_size=BATCH_SIZE,
        device="cuda",
    ):
        all_triples.append(triple)

print(f"Extracted {len(all_triples):,} triples before dedup")

# Deduplicate triples
seen = set()
unique_triples = []

for t in all_triples:
    key = (t["subject"], t["relation"], t["object"])
    if key not in seen:
        seen.add(key)
        unique_triples.append(t)

print(f"Unique triples: {len(unique_triples):,}")

Running REBEL on 5,000 batches (~1.9 hrs)...


REBEL batches: 100%|██████████| 5000/5000 [2:03:23<00:00,  1.48s/it]

Extracted 12,792 triples before dedup
Unique triples: 11,493


In [52]:
seen = set()
unique_triples = []

for t in all_triples:
    key = (t["subject"], t["relation"], t["object"])
    if key not in seen:
        seen.add(key)
        unique_triples.append(t)

print(f"Unique triples: {len(unique_triples):,}")

Unique triples: 11,493


In [53]:
TRIPLES_PATH = "/content/drive/MyDrive/graphbench/rebel_triples.parquet"

df = pd.DataFrame(unique_triples)
df.to_parquet(TRIPLES_PATH, index=False)

print(f"Saved {len(df):,} triples to {TRIPLES_PATH}")

print(df.head())

print("\nRelation distribution:")
print(df["relation"].value_counts().head(20))

Saved 11,493 triples to /content/drive/MyDrive/graphbench/rebel_triples.parquet
          subject relation     object
0       echosmith    genre  indie pop
1          sydney  sibling       noah
2            noah  sibling     sydney
3  graham sierota  sibling     sydney
4           jamie  sibling     sydney

Relation distribution:
relation
located_in_the_administrative_territorial_entity    1636
instance_of                                         1287
country                                             1063
publication_date                                     875
genre                                                772
part_of                                              671
sport                                                553
shares_border_with                                   506
contains_administrative_territorial_entity           440
cast_member                                          358
spouse                                               349
member_of                       

In [54]:
from graphbench.ingestion.rebel_loader import stream_triples
from graphbench.ingestion.neo4j_writer import write_triples

triples = list(
    stream_triples(
        preextracted_path=TRIPLES_PATH,
        max_triples=60_000
    )
)

print(f"Loaded {len(triples):,} triples from parquet")

print("Writing to Neo4j...")
n = write_triples(triples, neo4j, batch_size=500)

print(f"Wrote {n:,} triples | Neo4j count: {neo4j.count_triples():,}")

Loaded 11,493 triples from parquet
Writing to Neo4j...


Writing relation types: 100%|██████████| 47/47 [00:09<00:00,  4.93it/s]


Wrote 11,493 triples | Neo4j count: 11,493


In [55]:
# Get every entity name stored in Neo4j

with neo4j._driver.session() as session:
    result = session.run("MATCH (e:Entity) RETURN e.name AS name")
    entity_names = [r["name"] for r in result]

print(f"Entities to embed: {len(entity_names):,}")

Entities to embed: 13,146


In [56]:
from sentence_transformers import SentenceTransformer
import numpy as np

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

print(f"Loading {EMBEDDING_MODEL}...")
embedder = SentenceTransformer(EMBEDDING_MODEL, device="cuda")

print("Embedding entities...")

embeddings = embedder.encode(
    entity_names,
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True,  # required for cosine similarity via IP
    convert_to_numpy=True,
)

print(f"Embedding matrix shape: {embeddings.shape}")  # (n_entities, 384)

Loading sentence-transformers/all-MiniLM-L6-v2...
Embedding entities...


Batches:   0%|          | 0/52 [00:00<?, ?it/s]

Embedding matrix shape: (13146, 384)


In [57]:
import faiss
import os

FAISS_INDEX_PATH = "/content/drive/MyDrive/graphbench/faiss_index/graphbench.index"
ENTITY_NAMES_PATH = "/content/drive/MyDrive/graphbench/faiss_index/entity_names.npy"

os.makedirs(os.path.dirname(FAISS_INDEX_PATH), exist_ok=True)

# Build FAISS index
dim = embeddings.shape[1]  # 384

index = faiss.IndexFlatIP(dim)
index.add(embeddings.astype("float32"))

print(f"FAISS index size: {index.ntotal:,} vectors")

# Save index + entity mapping
faiss.write_index(index, FAISS_INDEX_PATH)
np.save(ENTITY_NAMES_PATH, np.array(entity_names))

print(f"Saved index to {FAISS_INDEX_PATH}")
print(f"Saved entity names to {ENTITY_NAMES_PATH}")

FAISS index size: 13,146 vectors
Saved index to /content/drive/MyDrive/graphbench/faiss_index/graphbench.index
Saved entity names to /content/drive/MyDrive/graphbench/faiss_index/entity_names.npy


In [58]:
import numpy as np
import faiss

index2 = faiss.read_index(FAISS_INDEX_PATH)
entity_names2 = np.load(ENTITY_NAMES_PATH, allow_pickle=True).tolist()

test_query = embedder.encode(
    ["marie curie"],
    normalize_embeddings=True
)

scores, indices = index2.search(test_query.astype("float32"), k=5)

print("Top-5 nearest entities to 'marie curie':")

for score, idx in zip(scores[0], indices[0]):
    print(f"{entity_names2[idx]:<30} score={score:.4f}")

Top-5 nearest entities to 'marie curie':
ste. marie                     score=0.6983
anne vernon                    score=0.6329
maria theresa                  score=0.6087
marienkirche                   score=0.5855
lisa marie presley             score=0.5844


## Phase 3 — GNN Training: 3-Layer GAT

In [59]:
from graphbench.ingestion.embedder import embed_entities

with neo4j._driver.session() as session:
      result = session.run("MATCH (e:Entity) RETURN e.name AS name")
      entity_names = [r["name"] for r in result]

print(f"Embedding {len(entity_names):,} entities...")
embeddings = embed_entities(entity_names, show_progress=True)
embedding_dict = dict(zip(entity_names, embeddings))
print(f"embedding_dict ready: {len(embedding_dict):,} entries, shape {embeddings.shape}")

Embedding 13,146 entities...


Batches:   0%|          | 0/52 [00:00<?, ?it/s]

embedding_dict ready: 13,146 entries, shape (13146, 384)


In [60]:
# Build KGDataset with 80/10/10 train/val/test split
from graphbench.gnn.dataset import KGDataset

print("Building KGDataset...")
ds = KGDataset(triples, embedding_dict)
train_data, val_data, test_data = ds.split()

print(f"Nodes       : {train_data.x.shape[0]:,}")
print(
    f"Train edges : {train_data.edge_label_index.shape[1]:,}  "
    f"(pos={int(train_data.edge_label.sum()):,}, "
    f"neg={(train_data.edge_label == 0).sum().item():,})"
)
print(f"Val edges   : {val_data.edge_label_index.shape[1]:,}")
print(f"Test edges  : {test_data.edge_label_index.shape[1]:,}")

Building KGDataset...
Nodes       : 13,146
Train edges : 18,388  (pos=9,194, neg=9,194)
Val edges   : 2,298
Test edges  : 2,300


In [62]:
# Train 3-layer GAT — target: test AUC-ROC > 0.75
from graphbench.gnn.model import GATModel
from graphbench.gnn.trainer import train_gnn

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device}")

model = GATModel()
results = train_gnn(
    model=model,
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    checkpoint_dir=CHECKPOINT_DIR,
    device=device,
    epochs=500,
    lr=1e-3,
    early_stopping_patience=100,
)

print(f"\nTest AUC-ROC   : {results['test_auc']:.4f}")
print(f"Test Loss      : {results['test_loss']:.4f}")
print(
    f"Best Val AUC   : {results['best_val_auc']:.4f}  (epoch {results['best_epoch']})"
)

if results["test_auc"] < 0.75:
    print("\n⚠️  AUC below 0.75 threshold — consider more epochs or lower LR.")
else:
    print("\n✅ AUC threshold met — proceed to Phase 4.")

Training on: cuda


epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
lr,████▄▄▄▄▄▄▄▄▄▄▄▄▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
test_auc,▁
test_loss,▁
train_loss,██▇▅▄▄▃▃▃▃▃▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_auc,▁▁▄▄▄▅▆▆▆▆▇▇████████████████████████████
val_loss,██▆▄▄▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,159
lr,0.0
test_auc,0.78138
test_loss,0.59185



Test AUC-ROC   : 0.7814
Test Loss      : 0.5918
Best Val AUC   : 0.7698  (epoch 59)

✅ AUC threshold met — proceed to Phase 4.


In [63]:
# Load best checkpoint and verify weights round-trip
from graphbench.utils.checkpoint import load_best_checkpoint

checkpoint, ckpt_path = load_best_checkpoint(CHECKPOINT_DIR)
print(f"Best checkpoint : {ckpt_path.name}")
print(f'Epoch           : {checkpoint["epoch"]}')
print(f'Val AUC         : {checkpoint["val_auc"]:.4f}')

# Restore model for pipeline use
gat_model = GATModel()
gat_model.load_state_dict(checkpoint["model_state_dict"])
gat_model.eval()
print("Model restored from checkpoint.")

Best checkpoint : gat_epoch0058_auc0.7698_20260311_013551.pt
Epoch           : 58
Val AUC         : 0.7698
Model restored from checkpoint.


## Phase 4 — Pipelines: GraphRAG & GNN-RAG

In [64]:
import os
from pathlib import Path

DRIVE_BASE = Path("/content/drive/MyDrive/graphbench")

faiss_dir = DRIVE_BASE / "faiss_index"

print("Files inside FAISS directory:")
print(os.listdir(faiss_dir))

Files inside FAISS directory:
['graphbench.index', 'entity_names.npy']


In [65]:
import faiss
import numpy as np
from graphbench.utils.faiss_client import FAISSClient

index = faiss.read_index(str(DRIVE_BASE / "faiss_index" / "graphbench.index"))
entity_names_loaded = np.load(
    DRIVE_BASE / "faiss_index" / "entity_names.npy", allow_pickle=True
).tolist()

id_map = {i: name for i, name in enumerate(entity_names_loaded)}

faiss_client = FAISSClient(index=index, id_map=id_map)

print(f"FAISSClient ready: {faiss_client.size:,} vectors")

FAISSClient ready: 13,146 vectors


In [66]:
# Initialise both pipelines
from graphbench.utils.llm_client import LLMClient
from graphbench.community.detector import CommunityDetector
from graphbench.pipelines.graphrag_pipeline import GraphRAGPipeline
from graphbench.pipelines.gnnrag_pipeline import GNNRAGPipeline

# LLM: Mistral-7B-Instruct-v0.2 in 4-bit on GPU
llm = LLMClient(backend="hf")

graphrag = GraphRAGPipeline(
    neo4j_client=neo4j,
    faiss_client=faiss_client,
    llm_client=llm,
)

gnnrag = GNNRAGPipeline(
    neo4j_client=neo4j,
    faiss_client=faiss_client,
    llm_client=llm,
    gat_model=gat_model,
    entity_embeddings=embedding_dict,
)

print("GraphRAG  :", graphrag.name)
print("GNN-RAG   :", gnnrag.name)

GraphRAG  : GraphRAG
GNN-RAG   : GNN-RAG


In [68]:
# Monkey-patch the fix directly — no reinstall needed
from graphbench.utils import neo4j_client as _nc

def _get_subgraph_fixed(self, entity_name, *, hops=None, directed=False):
    from graphbench.utils.config import settings

    k = hops if hops is not None else settings.subgraph_hops
    rel_pattern = f"-[*1..{k}]->" if directed else f"-[*1..{k}]-"

    cypher = f"""
    MATCH path = (seed:Entity {{name: $name}}){rel_pattern}(neighbor:Entity)
    UNWIND relationships(path) AS rel
    RETURN
        startNode(rel).name AS subject,
        type(rel)           AS relation,
        endNode(rel).name   AS object
    """

    results = self.execute_read(cypher, name=entity_name)
    return [(r["subject"], r["relation"], r["object"]) for r in results]


import types
neo4j.get_subgraph = types.MethodType(_get_subgraph_fixed, neo4j)

print("Patch applied.")

Patch applied.


In [70]:
# Step 1 — Patch LLM to load without quantization
import types
from graphbench.utils import llm_client as _lc

def _load_hf_no_quant(self):
    from transformers import pipeline as hf_pipeline
    import torch

    self._hf_pipe = hf_pipeline(
        "text-generation",
        model=self._model,
        device_map="auto",
        torch_dtype=torch.float16,
        max_new_tokens=self._max_new_tokens,
    )

_lc.LLMClient._load_hf_pipeline = _load_hf_no_quant
print("Patch applied: fp16, no quantization")

Patch applied: fp16, no quantization


In [71]:
# Step 2 — Re-create the LLM and pipelines
from graphbench.utils.llm_client import LLMClient
from graphbench.pipelines.graphrag_pipeline import GraphRAGPipeline
from graphbench.pipelines.gnnrag_pipeline import GNNRAGPipeline

llm = LLMClient(backend="hf")

graphrag = GraphRAGPipeline(
    neo4j_client=neo4j,
    faiss_client=faiss_client,
    llm_client=llm,
)

gnnrag = GNNRAGPipeline(
    neo4j_client=neo4j,
    faiss_client=faiss_client,
    llm_client=llm,
    gat_model=gat_model,
    entity_embeddings=embedding_dict,
)

print("Pipelines ready")

Pipelines ready


In [73]:
import types
from graphbench.utils import llm_client as _lc

def _load_hf_no_quant(self):
    from transformers import pipeline as hf_pipeline
    import torch
    self._hf_pipeline = hf_pipeline(
        "text-generation",
        model=self._model,
        device_map="auto",
        torch_dtype=torch.float16,
        max_new_tokens=self.max_new_tokens,
    )

_lc.LLMClient._load_hf_pipeline = _load_hf_no_quant
llm._hf_pipeline = None  # force reload on next call
print("Patch applied")

Patch applied


In [75]:
import types
from graphbench.utils import llm_client as _lc

def _load_hf_no_quant(self):
    from transformers import pipeline as hf_pipeline
    import torch
    pipe = hf_pipeline(
        "text-generation",
        model=self._model,
        device_map="auto",
        dtype=torch.float16,
        max_new_tokens=self.max_new_tokens,
    )
    return pipe  # must return — _generate_hf assigns the return value

_lc.LLMClient._load_hf_pipeline = _load_hf_no_quant
llm._hf_pipeline = None  # force reload
print("Patch applied")

Patch applied


In [76]:
# Smoke test on 5 questions
import time
import pandas as pd

smoke_questions = [
    "Where was Marie Curie born?",
    "What did Albert Einstein win the Nobel Prize for?",
    "Which country is Warsaw located in?",
    "Who discovered penicillin?",
    "In what field did Marie Curie work?",
]

rows = []
for q in smoke_questions:
    t0 = time.perf_counter()
    ra = graphrag.answer(q)
    lat_a = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    rb = gnnrag.answer(q)
    lat_b = (time.perf_counter() - t0) * 1000

    rows.append(
        {
            "Question": q,
            "GraphRAG Answer": ra.predicted_answer,
            "GraphRAG ms": f"{lat_a:.0f}",
            "GNN-RAG Answer": rb.predicted_answer,
            "GNN-RAG ms": f"{lat_b:.0f}",
        }
    )

df = pd.DataFrame(rows)
print(df.to_string(index=False))

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use cuda:0


                                         Question                                                                                                                                                                                                                                                                GraphRAG Answer GraphRAG ms                                                                                                                                                   GNN-RAG Answer GNN-RAG ms
                      Where was Marie Curie born?                                                                                                                                                                                                                     I don't know. Marie Curie is not mentioned in the context.        6447                                                                                                       I don't know. Marie Curie is not mentioned in the context.  

## Phase 5 — Benchmark: 500 HotpotQA Questions

In [77]:
# Run full benchmark — ~2-4 hours depending on GPU and LLM latency
from graphbench.benchmark.evaluator import Evaluator

evaluator = Evaluator(
    pipeline_a=graphrag,
    pipeline_b=gnnrag,
    n_questions=500,
    seed=42,
    results_dir=RESULTS_DIR,
)

summary = evaluator.run()
print("\nBenchmark complete.")

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

Benchmark: 100%|██████████| 500/500 [51:59<00:00,  6.24s/q]


GNN-RAG/em,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
GNN-RAG/f1,▁▁▁▁▁▁▂▁▁▂▁▁▁▃▁▁▁▁▃▁▁▄▄▃▁▃▅▁▄▁▁▆▆▅▄▂▅▁▄█
GNN-RAG/latency_ms,▂▂▂▂▂▁▃▃▂▇▃▃▂▂▂▃▃▁▂▁█▂▂▃▃▂▃▂▂▁▂▃▂▂▂▂▃▂▂▂
GraphRAG/em,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
GraphRAG/f1,▁▁▁▁▁▁▂▇▁▂▁▁▁▂▁▁▁▁▃▁▄▄▃▁▁▄▄▁▄▅▅▆▄▄▁▂▁▁▃█
GraphRAG/latency_ms,▄▅▃▃▂▄▅▁▃▆▄▂▄▅▃▃▄▁▃▂▆▃▃▃▄▅▃▂▂▃▁▅▅▅▄█▃▅▄▅
GNN-RAG/em,0.038
GNN-RAG/f1,0.10899
GNN-RAG/latency_ms,3054.03691
GNN-RAG/latency_p50,3044.52014
GNN-RAG/latency_p95,3888.28782



Benchmark complete.


In [78]:
# Summary table
import pandas as pd

rows = []
for pipeline_name, metrics in summary.items():
    rows.append(
        {
            "Pipeline": pipeline_name,
            "EM (%)": f"{metrics['em'] * 100:.2f}",
            "F1 (%)": f"{metrics['f1'] * 100:.2f}",
            "Latency P50 (ms)": f"{metrics['latency_p50']:.0f}",
            "Latency P95 (ms)": f"{metrics['latency_p95']:.0f}",
            "N Questions": metrics["n_questions"],
        }
    )

df_summary = pd.DataFrame(rows).set_index("Pipeline")
print("=== Benchmark Results ===")
print(df_summary.to_string())

=== Benchmark Results ===
         EM (%) F1 (%) Latency P50 (ms) Latency P95 (ms)  N Questions
Pipeline                                                             
GraphRAG   3.80  10.51             3068             3805          500
GNN-RAG    3.80  10.90             3045             3888          500


## Phase 6 — Results & Analysis

In [79]:
# Load the most recent results file
import json, glob
from pathlib import Path

result_files = sorted(Path(RESULTS_DIR).glob("*_results.json"))
assert result_files, f"No results found in {RESULTS_DIR}. Run Phase 5 first."

with open(result_files[-1]) as f:
    results = json.load(f)

summary = results["summary"]
rows_data = results["rows"]
print(f"Loaded {len(rows_data)} question results from {result_files[-1].name}")

Loaded 500 question results from 20260311_024515_results.json


In [80]:
# EM & F1 comparison bar chart
import plotly.graph_objects as go

pipelines = list(summary.keys())
em_vals = [summary[p]["em"] * 100 for p in pipelines]
f1_vals = [summary[p]["f1"] * 100 for p in pipelines]

fig = go.Figure()
fig.add_trace(
    go.Bar(
        name="Exact Match (%)",
        x=pipelines,
        y=em_vals,
        marker_color="#4e79a7",
        text=[f"{v:.1f}%" for v in em_vals],
        textposition="outside",
    )
)
fig.add_trace(
    go.Bar(
        name="Token F1 (%)",
        x=pipelines,
        y=f1_vals,
        marker_color="#f28e2b",
        text=[f"{v:.1f}%" for v in f1_vals],
        textposition="outside",
    )
)
fig.update_layout(
    title="GraphRAG vs GNN-RAG: Exact Match & F1",
    yaxis_title="Score (%)",
    yaxis_range=[0, 105],
    barmode="group",
    template="plotly_white",
)
fig.show()

In [81]:
# Latency comparison
p50_vals = [summary[p]["latency_p50"] for p in pipelines]
p95_vals = [summary[p]["latency_p95"] for p in pipelines]

fig2 = go.Figure()
fig2.add_trace(
    go.Bar(
        name="P50 (ms)",
        x=pipelines,
        y=p50_vals,
        marker_color="#59a14f",
        text=[f"{v:.0f} ms" for v in p50_vals],
        textposition="outside",
    )
)
fig2.add_trace(
    go.Bar(
        name="P95 (ms)",
        x=pipelines,
        y=p95_vals,
        marker_color="#e15759",
        text=[f"{v:.0f} ms" for v in p95_vals],
        textposition="outside",
    )
)
fig2.update_layout(
    title="Latency Comparison (ms)",
    yaxis_title="Latency (ms)",
    barmode="group",
    template="plotly_white",
)
fig2.show()

In [82]:
# Break down EM/F1 by question type (bridge vs comparison)
import pandas as pd
import numpy as np
from graphbench.benchmark.metrics import exact_match, token_f1

df_rows = pd.DataFrame(rows_data)

for pipeline_col_prefix in [p.replace("-", "_") for p in pipelines]:
    original_name = (
        pipeline_col_prefix.replace("_", "-")
        if "-" not in pipeline_col_prefix
        else pipeline_col_prefix
    )
    # find the matching column prefix

for pipeline in pipelines:
    pred_col = f"{pipeline}_predicted"
    if pred_col not in df_rows.columns:
        continue
    df_rows[f"{pipeline}_em"] = df_rows.apply(
        lambda r, p=pipeline: exact_match(r[f"{p}_predicted"], r["gold_answer"]), axis=1
    )
    df_rows[f"{pipeline}_f1"] = df_rows.apply(
        lambda r, p=pipeline: token_f1(r[f"{p}_predicted"], r["gold_answer"]), axis=1
    )

type_breakdown = []
for qtype in ["bridge", "comparison"]:
    subset = df_rows[df_rows["type"] == qtype]
    row = {"type": qtype, "n": len(subset)}
    for pipeline in pipelines:
        em_col = f"{pipeline}_em"
        f1_col = f"{pipeline}_f1"
        if em_col in subset.columns:
            row[f"{pipeline} EM"] = f"{subset[em_col].mean() * 100:.1f}%"
            row[f"{pipeline} F1"] = f"{subset[f1_col].mean() * 100:.1f}%"
    type_breakdown.append(row)

print("=== Results by Question Type ===")
print(pd.DataFrame(type_breakdown).to_string(index=False))

=== Results by Question Type ===
      type   n GraphRAG EM GraphRAG F1 GNN-RAG EM GNN-RAG F1
    bridge 250        4.8%        7.8%       4.4%       7.6%
comparison 250        2.8%       13.3%       3.2%      14.2%


In [83]:
# Show 5 most interesting error cases (high F1 for one, low for other)
if len(pipelines) >= 2:
    pa, pb = pipelines[0], pipelines[1]
    em_a_col = f"{pa}_em"
    em_b_col = f"{pb}_em"
    if em_a_col in df_rows.columns and em_b_col in df_rows.columns:
        # Cases where GNN-RAG wins but GraphRAG fails
        gnn_wins = df_rows[(df_rows[em_b_col] == 1) & (df_rows[em_a_col] == 0)]
        print(f"Questions where {pb} wins ({len(gnn_wins)} total):")
        for _, r in gnn_wins.head(3).iterrows():
            print(f'  Q: {r["question"][:80]}')
            print(
                f'     Gold: {r["gold_answer"]}  |  {pa}: {r[f"{pa}_predicted"]}  |  {pb}: {r[f"{pb}_predicted"]}'
            )

Questions where GNN-RAG wins (9 total):
  Q: Which 1947 play written by American playwright Tennessee Williams did Cate Blanc
     Gold: A Streetcar Named Desire  |  GraphRAG: I don't know.

The context provided does not mention any information about a 1947 play written by Tennessee Williams that Cate Blanchett acted in.  |  GNN-RAG: A Streetcar Named Desire
  Q: Xavier Saint-Just is famous for his paintings for a children's book by Felix Sal
     Gold: Walt Disney  |  GraphRAG: Tony Scott.  |  GNN-RAG: Walt Disney
  Q: Catherine Anne "Cat" Taber, is an American actress who has appeared in films, te
     Gold: The Loud House  |  GraphRAG: The Loud House animated series is not mentioned in the context. I don't know.  |  GNN-RAG: The Loud House.


In [84]:
# Write results to docs/benchmark_results.md
import textwrap
from datetime import datetime, timezone

md_lines = [
    "# GraphBench Benchmark Results",
    "",
    f'*Generated: {datetime.now(timezone.utc).strftime("%Y-%m-%d")}*',
    "",
    "## Setup",
    "- Dataset: HotpotQA distractor (500 questions: 250 bridge + 250 comparison)",
    "- KG: ~50k REBEL triples in Neo4j AuraDB Free",
    "- Embeddings: sentence-transformers/all-MiniLM-L6-v2 (384-dim)",
    "- LLM: mistralai/Mistral-7B-Instruct-v0.2 (4-bit on T4)",
    "- GNN: 3-layer GAT (4 heads), trained to AUC-ROC > 0.75",
    "",
    "## Results",
    "",
    "| Pipeline | EM (%) | F1 (%) | Latency P50 (ms) | Latency P95 (ms) |",
    "|----------|--------|--------|-----------------|-----------------|",
]

for pipeline, metrics in summary.items():
    md_lines.append(
        f'| {pipeline} | {metrics["em"]*100:.2f} | {metrics["f1"]*100:.2f} '
        f'| {metrics["latency_p50"]:.0f} | {metrics["latency_p95"]:.0f} |'
    )

md_lines += [
    "",
    "## Notes",
    "- Latency measured with `time.perf_counter()` around `pipeline.answer()` only.",
    "- EM and F1 computed with `normalize_answer()` (lowercase, remove articles/punctuation).",
]

md_content = "\n".join(md_lines)

# Save to docs/ in repo (relative path for Colab)
import subprocess

result = subprocess.run(
    ["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True
)
repo_root = Path(result.stdout.strip()) if result.returncode == 0 else Path(".")
md_path = repo_root / "docs" / "benchmark_results.md"
md_path.parent.mkdir(exist_ok=True)
md_path.write_text(md_content)
print(f"benchmark_results.md written to {md_path}")
print()
print(md_content)

benchmark_results.md written to docs/benchmark_results.md

# GraphBench Benchmark Results

*Generated: 2026-03-11*

## Setup
- Dataset: HotpotQA distractor (500 questions: 250 bridge + 250 comparison)
- KG: ~50k REBEL triples in Neo4j AuraDB Free
- Embeddings: sentence-transformers/all-MiniLM-L6-v2 (384-dim)
- LLM: mistralai/Mistral-7B-Instruct-v0.2 (4-bit on T4)
- GNN: 3-layer GAT (4 heads), trained to AUC-ROC > 0.75

## Results

| Pipeline | EM (%) | F1 (%) | Latency P50 (ms) | Latency P95 (ms) |
|----------|--------|--------|-----------------|-----------------|
| GraphRAG | 3.80 | 10.51 | 3068 | 3805 |
| GNN-RAG | 3.80 | 10.90 | 3045 | 3888 |

## Notes
- Latency measured with `time.perf_counter()` around `pipeline.answer()` only.
- EM and F1 computed with `normalize_answer()` (lowercase, remove articles/punctuation).


## Phase 7 — Export for Demo

In [85]:
# Save summary.json for HuggingFace Spaces leaderboard tab
import json

summary_path = RESULTS_DIR / "summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"summary.json → {summary_path}")

summary.json → /content/drive/MyDrive/graphbench/results/summary.json


In [87]:
print("=== Assets to upload to HuggingFace Spaces ===")

assets = [
    (DRIVE_BASE / "faiss_index" / "graphbench.index", "FAISS index"),
    (DRIVE_BASE / "faiss_index" / "entity_names.npy", "FAISS entity names"),
    (summary_path, "Benchmark summary"),
]

try:
    ckpt_path_demo = sorted(CHECKPOINT_DIR.glob("*.pt"))[-1]
    assets.append((ckpt_path_demo, "Best GAT checkpoint"))
except IndexError:
    pass

for path, label in assets:
    size_mb = path.stat().st_size / 1e6 if path.exists() else 0
    status = "✅" if path.exists() else "❌ MISSING"
    print(f"{status}  {label:30s}  {size_mb:.1f} MB   {path}")

=== Assets to upload to HuggingFace Spaces ===
✅  FAISS index                     20.2 MB   /content/drive/MyDrive/graphbench/faiss_index/graphbench.index
✅  FAISS entity names              5.4 MB   /content/drive/MyDrive/graphbench/faiss_index/entity_names.npy
✅  Benchmark summary               0.0 MB   /content/drive/MyDrive/graphbench/results/summary.json
✅  Best GAT checkpoint             1.5 MB   /content/drive/MyDrive/graphbench/checkpoints/gat_epoch0172_auc0.7697_20260311_011536.pt


In [100]:
print("🎉 All phases complete!")
print()
for pipeline, metrics in summary.items():
    print(
        f'{pipeline:10s}  EM={metrics["em"]*100:.2f}%  F1={metrics["f1"]*100:.2f}%'
        f'  P50={metrics["latency_p50"]:.0f}ms  P95={metrics["latency_p95"]:.0f}ms'
    )

🎉 All phases complete!

GraphRAG    EM=3.80%  F1=10.51%  P50=3068ms  P95=3805ms
GNN-RAG     EM=3.80%  F1=10.90%  P50=3045ms  P95=3888ms
